# Moteur de recherche documentaire

**Programmation de spécialité — Master 1 Informatique**

Interface graphique permettant d'interroger un corpus
de documents Reddit et Arxiv consacré au coronavirus.

Le moteur utilise :
- une représentation Documents × Termes ;
- une pondération TF-IDF ;
- la similarité cosinus ;
- un classement des résultats par pertinence.

Cette interface réutilise les classes Python du projet.

In [1]:
from pathlib import Path
from IPython.display import display, clear_output
import ipywidgets as widgets
from projet.corpus import Corpus
from projet.document import Document
from projet.search_engine import SearchEngine
from projet.text_processing import nettoyer_texte

print("Imports réussis.")

Imports réussis.


In [2]:
chemin_corpus = Path("data/corpus.csv")
if not chemin_corpus.is_file():
    raise FileNotFoundError(
        f"Corpus introuvable : {chemin_corpus.resolve()}"
    )
corpus = Corpus.get_instance("Coronavirus")
corpus.load(chemin_corpus)

print("Corpus chargé avec succès.")
print("Nombre de documents :", corpus.get_document_count())

Corpus chargé avec succès.
Nombre de documents : 1240


In [3]:
moteur = SearchEngine(
    corpus,
    show_progress=True
)
print("\nIndexation terminée.")
print(f"Documents : {len(moteur.document_ids)}")
print(f"Mots distincts : {len(moteur.vocab)}")
print(f"Matrice TF : {moteur.mat_TF.shape}")
print(f"Matrice TF-IDF : {moteur.mat_TFxIDF.shape}")

Préparation de l'index:   0%|          | 0/1240 [00:00<?, ?document/s]


Indexation terminée.
Documents : 1240
Mots distincts : 11631
Matrice TF : (1240, 11631)
Matrice TF-IDF : (1240, 11631)


In [4]:
requetes = [
    "covid vaccine",
    "coronavirus treatment",
    "vaccine research"
]
for requete in requetes:
    print(f"\nRecherche : {requete}")
    resultats = moteur.search(
        mots_cles=requete,
        n=5
    )
    if resultats.empty:
        print("Aucun document trouvé.")
    else:
        display(
            resultats.round({"score": 4})
        )


Recherche : covid vaccine


,document_id,title,author,source,score,url
0,467,Novavax's updated XBB.1.5 COVID-19 vaccine in ...,Womandarine,reddit,0.2970,https://ir.novavax.com/press-releases/2024-04-...
1,203,FDA appears to be slow-walking vaccine approva...,mepper,reddit,0.2527,https://www.nbcnews.com/health/health-news/fda...
2,1889,Estimating Vaccine Efficacy Over Time After a ...,Anastasios A. Tsiatis,arxiv,0.2275,http://arxiv.org/abs/2102.13103v1
3,809,Covid-19 vaccine effectiveness against post-co...,SteamedHamSalad,reddit,0.2138,https://www.bmj.com/content/383/bmj-2023-076990
4,291,"New COVID vaccine for 2024, isolation guidelin...",BothZookeepergame612,reddit,0.2136,https://www.cbsnews.com/news/covid-2024-guidel...



Recherche : coronavirus treatment


,document_id,title,author,source,score,url
0,41,Hyperbaric oxygen therapy is being explored as...,scientificamerican,reddit,0.2245,https://www.scientificamerican.com/article/hyp...
1,256,Controversial COVID study that promoted hydrox...,fat_cock_freddy,reddit,0.2165,https://www.nature.com/articles/d41586-024-040...
2,575,Why does COVID-19 make you sneeze? | Viral pro...,hexagonincircuit1594,reddit,0.1882,https://www.science.org/content/article/why-do...
3,1981,Determination of an Optimal Control Strategy f...,Gustavo Barbosa Libotte,arxiv,0.1787,http://arxiv.org/abs/2004.07397v2
4,763,Home Test to Treat program extends nationwide:...,hexagonincircuit1594,reddit,0.1739,https://www.nih.gov/news-events/news-releases/...



Recherche : vaccine research


,document_id,title,author,source,score,url
0,467,Novavax's updated XBB.1.5 COVID-19 vaccine in ...,Womandarine,reddit,0.2204,https://ir.novavax.com/press-releases/2024-04-...
1,203,FDA appears to be slow-walking vaccine approva...,mepper,reddit,0.1876,https://www.nbcnews.com/health/health-news/fda...
2,1889,Estimating Vaccine Efficacy Over Time After a ...,Anastasios A. Tsiatis,arxiv,0.1764,http://arxiv.org/abs/2102.13103v1
3,1922,Collaboration in the Time of COVID: A Scientom...,Eoghan Cunningham,arxiv,0.1673,http://arxiv.org/abs/2108.13370v1
4,1491,Towards reliable and transparent vaccine phase...,Ivan da Silva Sendin,arxiv,0.1598,http://arxiv.org/abs/2102.07022v1


In [5]:
titre = widgets.Label(
    value="Moteur de recherche"
)
champ_mots = widgets.Text(
    value="",
    placeholder="Exemple : covid vaccine",
    description="Mots clés :",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="600px")
)
curseur_nombre = widgets.IntSlider(
    value=5,
    min=1,
    max=20,
    step=1,
    description="Nombre d'articles à extraire :",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="550px"),
    continuous_update=False
)
bouton_rechercher = widgets.Button(
    description="Rechercher",
    button_style="primary",
    icon="search"
)
zone_resultats = widgets.Output()

filtre_source = widgets.Dropdown(
    options=[
        ("Toutes les sources", None),
        ("Reddit", "reddit"),
        ("Arxiv", "arxiv")
    ],
    value=None,
    description="Source :",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="300px")
)

In [6]:
def clique_bouton(bouton) -> None:
    mots_cles: str = champ_mots.value.strip()
    nombre_articles: int = curseur_nombre.value
    with zone_resultats:
        clear_output(wait=True)
        if not mots_cles:
            print("Veuillez saisir des mots-clés.")
            return
        resultats = moteur.search(
            mots_cles=mots_cles,
            n=nombre_articles,
            source=filtre_source.value
        )
        if resultats.empty:
            print("Aucun document pertinent trouvé.")
            return
        print(
            f"{len(resultats)} document(s) trouvé(s) "
            f"pour : {mots_cles}"
        )
        display(
            resultats.round({"score": 4})
        )

In [7]:
bouton_rechercher.on_click(clique_bouton)
ligne_actions = widgets.HBox([
    curseur_nombre,
    bouton_rechercher
])
interface = widgets.VBox([
    titre,
    champ_mots,
    ligne_actions,
    zone_resultats
])
display(interface)

In [8]:
selection_mot = widgets.Dropdown(
    options=[("Saisir une requête", None)],
    value=None,
    description="Mot à explorer :",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="400px")
)
def actualiser_mots(change) -> None:
    texte = nettoyer_texte(change["new"])
    mots = list(dict.fromkeys(texte.split()))
    selection_mot.options = [
        ("Choisir un mot", None)
    ] + [
        (mot, mot) for mot in mots
    ]
    
champ_mots.observe(
    actualiser_mots,
    names="value"
)
actualiser_mots({
    "new": champ_mots.value
})
bouton_concordancier = widgets.Button(
    description="Concordancier",
    button_style="info"
)

def clique_concordancier(bouton) -> None:
    mot = selection_mot.value
    with zone_resultats:
        clear_output(wait=True)
        if mot is None:
            print("Veuillez sélectionner un mot à explorer.")
            return
        concordances = corpus.concorde(
            expression=mot,
            context_size=40
        )
        if concordances.empty:
            print("Aucune concordance trouvée.")
            return
        print(f"Mot recherché : {mot}")
        print(
            "Concordancier sur l'ensemble du corpus "
            "(toutes les sources)."
        )
        print(
            f"{len(concordances)} occurrence(s) trouvée(s)."
        )
        display(concordances.head(20))

bouton_concordancier.on_click(
    clique_concordancier
)

ligne_actions.children = (
    curseur_nombre,
    bouton_rechercher,
    bouton_concordancier
)
interface.children = (
    titre,
    champ_mots,
    filtre_source,
    selection_mot,
    ligne_actions,
    zone_resultats
)